# 1. Build the bilingual Family Code corpus

Turns the scanned Amharic Revised Family Code (Proclamation No. 213/2000 · አዋጅ ቁጥር 213/1992) into one
record per article, matched with the English text of the same article.

- **Amharic:** the scan has no usable text layer, so every page is OCR'd with Tesseract's `amh` model
  (`tessdata_best`).
- **English:** the FAOLEX copy has a clean text layer, extracted with `pdftotext`.

**Runtime:** CPU is enough (about 5-10 minutes for 99 pages). Before running, upload the Amharic PDF
as `family_code_am.pdf` in the Colab file panel, or run the cell below and pick the file.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr poppler-utils > /dev/null
!pip -q install opencv-python-headless
# Most accurate Amharic model, kept in a local folder and passed with --tessdata-dir
!mkdir -p tessdata && wget -q -O tessdata/amh.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/amh.traineddata
!tesseract --version | head -1 && tesseract --tessdata-dir tessdata --list-langs | grep amh && pdftoppm -v 2>&1 | head -1
!rm -rf amharic-legal-rag && git clone -q https://github.com/4brhamy/amharic-legal-rag.git
import sys
sys.path.insert(0, "amharic-legal-rag")

In [ ]:
import os
if not os.path.exists("family_code_am.pdf"):
    from google.colab import files
    up = files.upload()
    os.rename(next(iter(up)), "family_code_am.pdf")
!wget -q -O family_code_en.pdf https://faolex.fao.org/docs/pdf/eth208224.pdf
!pdfinfo family_code_am.pdf | grep Pages && pdfinfo family_code_en.pdf | grep Pages

## OCR every Amharic page
Pages are rendered at 300 DPI. Two settings matter for this scan (found by testing on problem pages):
- **Underline removal.** Headings are underlined, which makes Tesseract misread them.
- **`--psm 6`** (one uniform block of text). The default layout analysis drops some underlined
  headings entirely, e.g. "Article 112 – Appeal" on page 41.

In [ ]:
import glob
from concurrent.futures import ThreadPoolExecutor
from amlegal.ocr import ocr_image

os.makedirs("pages", exist_ok=True)
os.makedirs("ocr", exist_ok=True)
!pdftoppm -r 300 -gray -png family_code_am.pdf pages/p

pngs = sorted(glob.glob("pages/*.png"))
with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    pages_am = list(pool.map(lambda p: ocr_image(p, psm=6, tessdata_dir="tessdata"), pngs))
for png, text in zip(pngs, pages_am):
    open(os.path.join("ocr", os.path.basename(png)[:-4] + ".txt"), "w", encoding="utf-8").write(text)
print(len(pages_am), "pages OCR'd")
print(pages_am[9][:800])

## Split into articles and check the result
The law has 323 articles. OCR occasionally misreads an Ethiopic numeral; the splitter checks every
heading against the running sequence and flags any number it had to infer.

In [ ]:
from amlegal.corpus import clean_page, split_articles, split_english, save_jsonl

arts_am = split_articles([clean_page(p) for p in pages_am])
nums = [a.number for a in arts_am]
missing = [n for n in range(1, 324) if n not in nums]
unchecked = [a.number for a in arts_am if not a.number_checked]
print(f"Amharic articles: {len(arts_am)} (last = {nums[-1] if nums else None})")
print(f"Missing numbers ({len(missing)}): {missing}")
print(f"Numbers inferred from the sequence ({len(unchecked)}): {unchecked}")

!pdftotext -layout family_code_en.pdf family_code_en.txt
arts_en = split_english(open("family_code_en.txt", encoding="utf-8").read())
print(f"English articles: {len(arts_en)}")

## Merge into one bilingual record per article

In [ ]:
en = {a.number: a for a in arts_en}
am = {a.number: a for a in arts_am}
records = []
for n in range(1, 324):
    a, e = am.get(n), en.get(n)
    records.append({
        "article": n,
        "title_am": a.title if a else "",
        "text_am": a.text if a else "",
        "title_en": e.title if e else "",
        "text_en": e.text if e else "",
        "page_am": a.page if a else None,
        "number_checked": a.number_checked if a else False,
    })

os.makedirs("out", exist_ok=True)
save_jsonl(records, "out/family_code.jsonl")
both = sum(1 for r in records if r["text_am"] and r["text_en"])
print(f"{both}/323 articles have both Amharic and English text")

import random
for r in random.sample(records, 3):
    print(f"\n--- Article {r['article']} ---\n{r['title_am']}\n{r['text_am'][:300]}\n\n{r['title_en']}\n{r['text_en'][:300]}")

## Save the outputs
`out/family_code.jsonl` is the corpus used by the rest of the project. The raw OCR pages are kept
too, so the OCR can be inspected or corrected later.

In [ ]:
!cp -r ocr out/ocr_pages && cd out && zip -qr ../corpus_outputs.zip . && cd .. && ls -la corpus_outputs.zip